# CODI vs KaVa — post-training comparison and Phase 3 ablations

This notebook is separate from training. It reads the completed step-96,405 CODI and KaVa checkpoints from Google Drive, creates a strict paired report from their saved predictions, and runs causal latent-state interventions without changing either checkpoint.

The default path runs the inexpensive existing-results comparison and the 200-example **KaVa** ablation gate. Full baseline evaluation and CODI ablations are opt-in switches. Evaluation runs in an active Colab cell; after output begins, the browser may be closed, but Colab can still terminate managed runtimes. Completed dataset files and intervention runs are mirrored to Drive while the cell is active.

In [ ]:
# Repository and durable-storage settings.
REPO_URL = "https://github.com/0x0shephard/latent-reasoning.git"
RUN_COMMIT = "main"  # Prefer the exact pushed commit SHA for repeatability.
REPO_DIR = "/content/latent-reasoning"
DRIVE_ROOT = "/content/drive/MyDrive/CODI_KAVA"
LOCAL_ROOT = "/content/codikava_runtime"

# Safe defaults: KaVa first, capped at the same examples as the saved baseline.
RUN_KAVA_ABLATIONS = True
RUN_CODI_ABLATIONS = False
ABLATION_LIMIT = 200  # Per dataset; MultiArith contains only 180 examples.
RUN_KAVA_POSITION_SWEEP = False  # Enable after both all-state reports are saved.
RUN_MATCHED_BATCH_DID = False  # Rerun batch-dependent controls at batch size 8.
MATCHED_BATCH_SIZE = 8
POSITION_SWEEP_PERMUTATION_SEED = 1000
RUN_FULL_BASELINES = False  # Expensive; run only after the capped analysis is saved.
BOOTSTRAP_SAMPLES = 10000


## 1. Mount Drive and check out the analysis code

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import datetime, json, os, pathlib, subprocess, sys, time
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"
pathlib.Path(DRIVE_ROOT).mkdir(parents=True, exist_ok=True)

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
target = "origin/main" if RUN_COMMIT == "main" else RUN_COMMIT
subprocess.run(["git", "-C", REPO_DIR, "checkout", "--detach", target], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", os.path.join(REPO_DIR, "requirements.txt")], check=True)
os.chdir(REPO_DIR)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Checked out:", commit)
if RUN_COMMIT == "main":
    print("For repeatability, replace RUN_COMMIT with:", commit)


## 2. Verify the GPU and final Drive artifacts

This checks archive structure and the saved evaluation summaries. The evaluation runner performs the full PyTorch payload verification before loading a checkpoint.

In [ ]:
import torch
assert torch.cuda.is_available(), "Select Runtime > Change runtime type > GPU"
print("Python:", sys.version.split()[0])
print("Torch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))

from scripts.colab_runner import validate_torch_checkpoint_archive
FINAL_STEP = 96405
for method in ("codi", "kava"):
    root = pathlib.Path(DRIVE_ROOT) / "outputs" / method
    checkpoint = root / "checkpoints" / f"step_{FINAL_STEP:08d}.pt"
    summary_path = root / "eval" / f"step_{FINAL_STEP:08d}" / "summary.json"
    validate_torch_checkpoint_archive(checkpoint)
    if not summary_path.is_file():
        raise FileNotFoundError(f"Missing saved evaluation: {summary_path}")
    summary = json.loads(summary_path.read_text())
    if int(summary.get("checkpoint_step", -1)) != FINAL_STEP:
        raise ValueError(f"Unexpected {method} evaluation step: {summary}")
    print(method.upper(), f"checkpoint={checkpoint.stat().st_size / 2**30:.2f} GiB", json.dumps(summary, indent=2))

subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=REPO_DIR, check=True)


In [ ]:
# Stream a long command to both this cell and a persistent Drive log.
def run_persisted(cmd, log_name):
    logs = pathlib.Path(DRIVE_ROOT) / "logs"
    logs.mkdir(parents=True, exist_ok=True)
    log_path = logs / log_name
    print("Starting:", " ".join(map(str, cmd)), flush=True)
    print("Persistent log:", log_path, flush=True)
    with log_path.open("a", encoding="utf-8", buffering=1) as log:
        log.write(f"\n=== {datetime.datetime.now(datetime.timezone.utc).isoformat()} {' '.join(map(str, cmd))} ===\n")
        process = subprocess.Popen(cmd, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        last_flush = time.monotonic()
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            if time.monotonic() - last_flush >= 30:
                log.flush()
                last_flush = time.monotonic()
        code = process.wait()
        log.flush()
    if code != 0:
        raise subprocess.CalledProcessError(code, cmd)
    return code


## 3. Compare the existing final CODI and KaVa evaluations

This is CPU-only and should finish quickly. It refuses different example sets instead of silently comparing incompatible dataset snapshots. The JSON report and readable Markdown table persist under `MyDrive/CODI_KAVA/reports/`.

In [ ]:
from IPython.display import Markdown, display
reports = pathlib.Path(DRIVE_ROOT) / "reports"
reports.mkdir(parents=True, exist_ok=True)
eval_dirs = {method: pathlib.Path(DRIVE_ROOT) / "outputs" / method / "eval" / f"step_{FINAL_STEP:08d}" for method in ("codi", "kava")}
comparison_path = reports / "codi_vs_kava_existing.json"
cmd = [
    sys.executable, "scripts/analyze_phase2.py",
    "--run", f"codi={eval_dirs['codi']}",
    "--run", f"kava={eval_dirs['kava']}",
    "--output", str(comparison_path),
    "--bootstrap-samples", str(BOOTSTRAP_SAMPLES),
]
subprocess.run(cmd, cwd=REPO_DIR, check=True)
display(Markdown(comparison_path.with_suffix(".md").read_text()))


## 4. Run the 200-example causal ablation gate

Each selected method runs three interventions: all latent states are zeroed, replaced by the batch mean, or deterministically shuffled across examples. The intervention is applied before each latent slot enters the transformer, so it changes the KV cache and downstream answer.

KaVa is enabled by default because it is the primary Phase-3 target. Enable CODI in the settings cell only after the KaVa gate succeeds. Results are written beneath `eval/step_00096405/ablations/`.

In [ ]:
selected_methods = []
if RUN_KAVA_ABLATIONS:
    selected_methods.append("kava")
if RUN_CODI_ABLATIONS:
    selected_methods.append("codi")
if not selected_methods:
    print("No ablations selected; change a RUN_*_ABLATIONS switch in the settings cell.")
for method in selected_methods:
    cmd = [
        sys.executable, "-u", "scripts/colab_ablation_runner.py",
        "--method", method,
        "--drive-root", DRIVE_ROOT,
        "--local-root", LOCAL_ROOT,
        "--limit", str(ABLATION_LIMIT),
    ]
    run_persisted(cmd, f"{method}_phase3_ablation.log")


## 5. Analyze each completed ablation

Run this before any optional full baseline evaluation: the existing baseline and the default interventions are all capped at the same 200 examples.

In [ ]:
for method in selected_methods:
    baseline = eval_dirs[method]
    ablations = baseline / "ablations"
    specs = [
        f"baseline={baseline}",
        f"zero={ablations / 'zero_all'}",
        f"mean={ablations / 'batch_mean_all'}",
        f"shuffle={ablations / 'batch_shuffle_all'}",
    ]
    output = reports / f"{method}_latent_ablation_limit{ABLATION_LIMIT}.json"
    cmd = [sys.executable, "scripts/analyze_phase2.py"]
    for spec in specs:
        cmd.extend(["--run", spec])
    cmd.extend(["--output", str(output), "--bootstrap-samples", str(BOOTSTRAP_SAMPLES)])
    subprocess.run(cmd, cwd=REPO_DIR, check=True)
    display(Markdown(output.with_suffix(".md").read_text()))


## 6. KaVa shuffle sweep over latent positions 0–5

Enable `RUN_KAVA_POSITION_SWEEP` in the settings cell. Each run shuffles exactly one latent position. The base seed is adjusted so every position uses the same actual cross-example permutation (`base seed + position = POSITION_SWEEP_PERMUTATION_SEED`), isolating position rather than permutation luck. All runs explicitly use batch size 8 and receive non-overwriting `_bs8` tags.

In [ ]:
if RUN_KAVA_POSITION_SWEEP:
    kava_baseline = eval_dirs["kava"]
    kava_ablations = kava_baseline / "ablations"
    for position in range(6):
        # LatentAblation uses seed + step; this keeps the realized permutation fixed.
        base_seed = POSITION_SWEEP_PERMUTATION_SEED - position
        cmd = [
            sys.executable, "-u", "scripts/colab_ablation_runner.py",
            "--method", "kava",
            "--drive-root", DRIVE_ROOT,
            "--local-root", LOCAL_ROOT,
            "--mode", "batch_shuffle",
            "--positions", str(position),
            "--limit", str(ABLATION_LIMIT),
            "--batch-size", str(MATCHED_BATCH_SIZE),
            "--seed", str(base_seed),
        ]
        run_persisted(cmd, f"kava_shuffle_position_{position}.log")
    sweep_output = reports / f"kava_shuffle_position_sweep_limit{ABLATION_LIMIT}.json"
    cmd = [
        sys.executable, "scripts/analyze_position_sweep.py",
        "--baseline", str(kava_baseline),
        "--output", str(sweep_output),
        "--bootstrap-samples", str(BOOTSTRAP_SAMPLES),
    ]
    for position in range(6):
        path = kava_ablations / f"batch_shuffle_p{position}_bs{MATCHED_BATCH_SIZE}"
        cmd.extend(["--position", f"{position}={path}"])
    subprocess.run(cmd, cwd=REPO_DIR, check=True)
    display(Markdown(sweep_output.with_suffix(".md").read_text()))
else:
    print("Skipped position sweep. Set RUN_KAVA_POSITION_SWEEP = True when ready.")


## 7. Direct CODI-vs-KaVa difference in differences

Zeroing is batch-independent and can be compared immediately using the existing `zero_all` runs. Mean and shuffle depend on batch composition: enable `RUN_MATCHED_BATCH_DID` to rerun a tagged baseline plus those controls for both methods at batch size 8 before comparing them. A negative DID means KaVa is harmed more than CODI.

In [ ]:
if RUN_MATCHED_BATCH_DID:
    for method in ("codi", "kava"):
        cmd = [
            sys.executable, "-u", "scripts/colab_ablation_runner.py",
            "--method", method,
            "--drive-root", DRIVE_ROOT,
            "--local-root", LOCAL_ROOT,
            "--mode", "baseline",
            "--mode", "batch_mean",
            "--mode", "batch_shuffle",
            "--limit", str(ABLATION_LIMIT),
            "--batch-size", str(MATCHED_BATCH_SIZE),
        ]
        run_persisted(cmd, f"{method}_matched_batch_ablation.log")

did_conditions = {"zero": (None, "zero_all")}
if RUN_MATCHED_BATCH_DID:
    did_conditions.update({
        "mean_bs8": (f"baseline_bs{MATCHED_BATCH_SIZE}", f"batch_mean_all_bs{MATCHED_BATCH_SIZE}"),
        "shuffle_bs8": (f"baseline_bs{MATCHED_BATCH_SIZE}", f"batch_shuffle_all_bs{MATCHED_BATCH_SIZE}"),
    })
for condition, (baseline_folder, intervention_folder) in did_conditions.items():
    codi_baseline = eval_dirs["codi"] if baseline_folder is None else eval_dirs["codi"] / "ablations" / baseline_folder
    kava_baseline = eval_dirs["kava"] if baseline_folder is None else eval_dirs["kava"] / "ablations" / baseline_folder
    output = reports / f"did_kava_minus_codi_{condition}_limit{ABLATION_LIMIT}.json"
    cmd = [
        sys.executable, "scripts/analyze_intervention_effects.py",
        "--left-name", "codi",
        "--left-baseline", str(codi_baseline),
        "--left-intervention", str(eval_dirs["codi"] / "ablations" / intervention_folder),
        "--right-name", "kava",
        "--right-baseline", str(kava_baseline),
        "--right-intervention", str(eval_dirs["kava"] / "ablations" / intervention_folder),
        "--intervention-name", condition,
        "--output", str(output),
        "--bootstrap-samples", str(BOOTSTRAP_SAMPLES),
    ]
    subprocess.run(cmd, cwd=REPO_DIR, check=True)
    display(Markdown(output.with_suffix(".md").read_text()))


## 8. Optional: full baseline evaluation

This is deliberately last because it overwrites each method's root prediction JSONLs with the full benchmark. The capped ablation reports above remain preserved. Set `RUN_FULL_BASELINES = True` only when you are ready for the longer evaluation.

In [ ]:
if RUN_FULL_BASELINES:
    for method in ("codi", "kava"):
        cmd = [
            sys.executable, "-u", "scripts/colab_ablation_runner.py",
            "--method", method,
            "--drive-root", DRIVE_ROOT,
            "--local-root", LOCAL_ROOT,
            "--mode", "baseline",
            "--limit", "0",
        ]
        run_persisted(cmd, f"{method}_full_eval.log")
    full_output = reports / "codi_vs_kava_full.json"
    cmd = [
        sys.executable, "scripts/analyze_phase2.py",
        "--run", f"codi={eval_dirs['codi']}",
        "--run", f"kava={eval_dirs['kava']}",
        "--output", str(full_output),
        "--bootstrap-samples", str(BOOTSTRAP_SAMPLES),
    ]
    subprocess.run(cmd, cwd=REPO_DIR, check=True)
    display(Markdown(full_output.with_suffix(".md").read_text()))
else:
    print("Skipped full evaluation. Set RUN_FULL_BASELINES = True when ready.")


## 9. Inspect durable status and reports

In [ ]:
for method in ("codi", "kava"):
    status_path = pathlib.Path(DRIVE_ROOT) / "status" / f"{method}.json"
    print(f"\n{method.upper()} STATUS")
    print(json.dumps(json.loads(status_path.read_text()), indent=2) if status_path.is_file() else "No status file")
print("\nREPORTS")
for path in sorted(reports.glob("*")):
    if path.is_file():
        print(f"{path.name}  {path.stat().st_size / 2**10:.1f} KiB")


## Decision gate

- If zero/mean/shuffle materially reduce KaVa accuracy with paired confidence intervals below zero, the latent block is causally carrying example-specific information. Run CODI next, then selected positions such as `--positions 0` and `--positions 5`.
- If interventions barely change accuracy, do not launch position sweeps yet; inspect generations and confirm the answer path depends on the latent cache.
- Full evaluation closes the seed-zero primary comparison, but the final research claim still requires `latent_nodistill`, `kava_random`, `kava_uniform`, and additional seeds.